In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("--- ANÁLISIS DE LA INFLUENCIA DE 'DURATION' EN 'DEPOSIT' ---\n")

# 1. Filtramos los clientes de "primera llamada absoluta"
condicion = (df['previous'] == 0) & (df['campaign'] == 1)
df_primera = df[condicion].copy()

print(f"Total de clientes analizados (puerta fría pura): {len(df_primera)}\n")

# =====================================================================
# TABLA 1: Estadísticos descriptivos básicos
# =====================================================================
print("1. RESUMEN ESTADÍSTICO DE DURACIÓN (segundos):")
tabla_stats = df_primera.groupby('deposit')['duration'].describe()[['count', 'mean', '50%', 'min', 'max']]
tabla_stats.columns = ['Volumen', 'Media (s)', 'Mediana (s)', 'Mínimo (s)', 'Máximo (s)']
display(tabla_stats.round(1))

# =====================================================================
# TABLA 2: Tasa de conversión agrupada por tramos de tiempo
# =====================================================================
# Cortamos la duración en intervalos lógicos: <1 min, 1-3 min, 3-5 min, 5-10 min, +10 min
bins = [0, 60, 180, 300, 600, float('inf')]
labels = ['< 1 min', '1 - 3 min', '3 - 5 min', '5 - 10 min', '> 10 min']
df_primera['rango_tiempo'] = pd.cut(df_primera['duration'], bins=bins, labels=labels, right=False)

# Estandarizamos la columna 'deposit' a numérico (0 y 1) para poder sumar los éxitos
if df_primera['deposit'].dtype == 'object':
    df_primera['deposit_num'] = (df_primera['deposit'] == 'yes').astype(int)
else:
    df_primera['deposit_num'] = df_primera['deposit']

tabla_rangos = df_primera.groupby('rango_tiempo', observed=True).agg(
    Total_Llamadas=('deposit_num', 'count'),
    Exitos_SI=('deposit_num', 'sum')
).reset_index()

# Calculamos el % de éxito por tramo
tabla_rangos['Tasa de Éxito (%)'] = (tabla_rangos['Exitos_SI'] / tabla_rangos['Total_Llamadas'] * 100).fillna(0)

print("\n2. TASA DE CONVERSIÓN POR TRAMOS DE DURACIÓN:")
display(tabla_rangos.round(2))


# =====================================================================
# GRÁFICOS: Distribución de la duración según el éxito
# =====================================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.set_style("whitegrid")

# Gráfico A: Boxplot (Cajas y Bigotes)
sns.boxplot(data=df_primera, x='deposit', y='duration', palette='Set2', ax=axes[0])
axes[0].set_title('Distribución de Duración según Decisión (Boxplot)', fontweight='bold', fontsize=14)
axes[0].set_xlabel('¿Contrata el depósito?', fontsize=12)
axes[0].set_ylabel('Duración de la llamada (segundos)', fontsize=12)

# Gráfico B: Histograma con curva de densidad (KDE)
sns.histplot(data=df_primera, x='duration', hue='deposit', kde=True, palette='Set2', ax=axes[1], bins=40, alpha=0.6)
axes[1].set_title('Volumen de Llamadas por Duración', fontweight='bold', fontsize=14)
axes[1].set_xlabel('Duración de la llamada (segundos)', fontsize=12)
axes[1].set_ylabel('Número de clientes', fontsize=12)
axes[1].set_xlim(0, 2000) # Limitamos a 2000s (33 min) para no aplastar el gráfico con outliers extremos

plt.tight_layout()
plt.show()

print("\n[!] CONCLUSIÓN VISUAL PARA NEGOCIO:")
print("Si observas la Tasa de Éxito por tramos, verás que las llamadas cortas (<3 min) tienen un éxito cercano a cero,")
print("mientras que las llamadas largas (>10 min) tienen tasas de éxito brutales. En los gráficos, la 'caja' de los que")
print("SÍ contratan está desplazada masivamente hacia arriba.")
print("Esto justifica matemáticamente la eliminación de la columna 'duration' para evitar el Data Leakage.")